In [ ]:
from tensorflow.keras import layers
from tensorflow import keras
import tensorflow as tf

from sklearn.model_selection import train_test_split

from ast import literal_eval
# is used for safely evaluating strings containing Python literals or container displays
# (e.g., lists, dictionaries) to their corresponding Python obb[poiuy1`234e5rt6y7i90o-p=[2wszjects.

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

df = pd.read_csv(""C:\Users\histe\Desktop\Recommendation System\arxiv_data_210930-054931.csv"")

df.head()

# ***DATA CLEANING AND PREPROCESSING***

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# getting unique labels
from ast import literal_eval
labels_column = df['terms'].apply(literal_eval)
labels = labels_column.explode().unique()
print("labels :",labels)
print("length :",len(labels))

In [ ]:
# remove duplicate entries based on the "titles" (terms) column
# This filters the DataFrame, keeping only the rows where the titles are not duplicated.
df = df[~df['titles'].duplicated()]
print(f"There are {len(df)} rows in the deduplicated dataset.")
# There are some terms with occurrence as low as 1.
print(sum(df['terms'].value_counts()==1))
# how many unique terms
print(df['terms'].nunique())

In [ ]:
# Filtering the rare terms. (it keeps only those rows where the "terms" value occurs more than once in the original DataFrame.)
df_filtered = df.groupby('terms').filter(lambda x: len(x) > 1)
df_filtered.shape

In [ ]:
# It evaluates the given string containing a Python literal or container display (e.g., a list or dictionary) and returns the corresponding Python object.
df_filtered['terms'] = df_filtered['terms'].apply(lambda x: literal_eval(x))
df_filtered['terms'].values[:3]

# ***Train Test slpit***

In [ ]:
from sklearn.model_selection import train_test_split
test_split = 0.1

# Initial train and test split.
# The stratify parameter ensures that the splitting is done in a way that preserves the same distribution of labels (terms) in both the training and test sets.
train_df, test_df = train_test_split(df_filtered,test_size=test_split,stratify=df_filtered["terms"].values,)

# Splitting the test set further into validation
# and new test sets.
val_df = test_df.sample(frac=0.5)
test_df.drop(val_df.index, inplace=True)

print(f"Number of rows in training set: {len(train_df)}")
print(f"Number of rows in validation set: {len(val_df)}")
print(f"Number of rows in test set: {len(test_df)}")

In [ ]:
# creates a TensorFlow RaggedTensor (terms) from the values in the "terms" column of the train_df DataFrame. A RaggedTensor is a tensor with non-uniform shapes
terms = tf.ragged.constant(train_df['terms'].values)
# This line creates a StringLookup layer in TensorFlow. The purpose of this layer is to map strings to integer indices and vice versa. The output_mode="multi_hot" indicates that the layer will output a multi-hot encoded representation of the input strings.
lookup = tf.keras.layers.StringLookup(output_mode='multi_hot')
# This step adapts the StringLookup layer to the unique values in the "terms" column, building the vocabulary.
lookup.adapt(terms)
# retrieve vocabulary
vocab = lookup.get_vocabulary()

print("Vocabulary:\n")
print(vocab)

In [ ]:
sample_label = train_df["terms"].iloc[0]
print(f"Original label: {sample_label}")

label_binarized = lookup([sample_label])
print(f"Label-binarized representation: {label_binarized}")

In [ ]:
# following lines::
# which is used for automatic adjustment of resource usage by TensorFlow's data loading pipeline.

#max_seqlen: Maximum sequence length. It indicates the maximum length allowed for sequences.
max_seqlen = 150
#batch_size: Batch size. It specifies the number of samples to use in each iteration.
batch_size = 128
#padding_token: A token used for padding sequences.
padding_token = "<pad>"
#auto = tf.data.AUTOTUNE: auto is assigned the value tf.data.AUTOTUNE,
auto = tf.data.AUTOTUNE

def make_dataset(dataframe, is_train=True):
    # creating sequences of labesls
    labels = tf.ragged.constant(dataframe["terms"].values)
    #This line uses the previously defined lookup layer to convert the ragged tensor of labels into a binarized representation. The resulting label_binarized is a NumPy array.
    label_binarized = lookup(labels).numpy()
    # creating sequences of text.
    dataset = tf.data.Dataset.from_tensor_slices((dataframe["abstracts"].values, label_binarized))
    # shuffling data basis on condition
    dataset = dataset.shuffle(batch_size * 10) if is_train else dataset
    return dataset.batch(batch_size)

"""
In summary, the make_dataset function is designed to create a
dataset suitable for training a model. It takes a dataframe as input,
assumes it has "abstracts" and "terms" columns, and creates a dataset of
batches where each batch consists of abstract
sequences and their corresponding binarized label sequences.
"""

In [21]:
train_dataset = make_dataset(train_df, is_train=True)
validation_dataset = make_dataset(val_df, is_train=False)
test_dataset = make_dataset(test_df, is_train=False)

In [ ]:
# Define the invert_multi_hot function
def invert_multi_hot(multi_hot_labels):
    """
    Converts a multi-hot encoded label back to its original terms.

    Args:
        multi_hot_labels: A multi-hot encoded label (numpy array).

    Returns:
        A list of original terms.
    """
    # Use the lookup layer's inverse vocabulary lookup
    return [vocab[i] for i, hot in enumerate(multi_hot_labels) if hot == 1]

# This code snippet is iterating through batches of the training dataset and printing the abstract text along with the corresponding labels.
text_batch, label_batch = next(iter(train_dataset))
for i, text in enumerate(text_batch[:5]):
    label = label_batch[i].numpy()[None, ...]
    print(f"Abstract: {text}")
    print(f"Label(s): {invert_multi_hot(label[0])}")
    print(" ")

In [ ]:
# This code calculates the size of the vocabulary in the "abstracts" column of the train_df DataFrame.

# Creating vocabulary with uniques words
vocabulary = set()
train_df["abstracts"].str.lower().str.split().apply(vocabulary.update)
vocabulary_size = len(vocabulary)
print(vocabulary_size)

# ***Text Vectorisation***

In [24]:
# Initializes a TextVectorization layer
text_vectorizer = layers.TextVectorization(max_tokens=vocabulary_size,ngrams=2,output_mode="tf_idf")
# `TextVectorization` layer needs to be adapted as per the vocabulary from our
# training set.
text_vectorizer.adapt(train_dataset.map(lambda text, label: text))

In [25]:
"""
Mapping Vectorization to Datasets: The code maps the text vectorization operation to
each element of the training, validation, and test datasets. This ensures that the text
data in each dataset is transformed into numerical vectors using the adapted TextVectorization layer.
The num_parallel_calls parameter is used to parallelize the mapping process, and prefetch is
applied to prefetch data batches
for better performance.
"""
train_dataset = train_dataset.map(lambda text, label: (text_vectorizer(text), label), num_parallel_calls=auto).prefetch(auto)
validation_dataset = validation_dataset.map(lambda text, label: (text_vectorizer(text), label), num_parallel_calls=auto).prefetch(auto)
test_dataset = test_dataset.map(lambda text, label: (text_vectorizer(text), label), num_parallel_calls=auto).prefetch(auto)

# ***Model Training***

In [ ]:
# creating shallow_mlp_model  (MLP)
from tensorflow.keras.callbacks import EarlyStopping

# Creating shallow_mlp_model (MLP) with dropout layers
model1 = keras.Sequential([
    # First hidden layer: 512 neurons, ReLU activation function, with dropout.
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.5),  # Adding dropout for regularization.

    # Second hidden layer: 256 neurons, ReLU activation function, with dropout.
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),  # Adding dropout for regularization.

    # Output layer: The number of neurons equals the vocabulary size (output vocabulary of the StringLookup layer), with a sigmoid activation function.
    layers.Dense(lookup.vocabulary_size(), activation='sigmoid')
])

# Compile the model
model1.compile(loss="binary_crossentropy", optimizer='adam', metrics=['binary_accuracy'])

# Add early stopping
# Number of epochs with no improvement after which training will be stopped.
# Restore weights from the epoch with the best value of the monitored quantity.
early_stopping = EarlyStopping(patience=5,restore_best_weights=True)

# Train the model
# Add early stopping callback.verbose=1
history = model1.fit(train_dataset,validation_data=validation_dataset,epochs=10,callbacks=[early_stopping])

In [ ]:
# plotting loss
def plot_result(item):
    plt.plot(history.history[item], label=item)
    plt.plot(history.history["val_" + item], label="val_" + item)
    plt.xlabel("Epochs")
    plt.ylabel(item)
    plt.title("Train and Validation {} Over Epochs".format(item), fontsize=14)
    plt.legend()
    plt.grid()
    plt.show()


plot_result("loss")
plot_result("binary_accuracy")

# `***Model Evaluation***`

In [ ]:
# model evaltuation on test and val dataset
_, binary_acc1 = model1.evaluate(test_dataset)
_, binary_acc2 = model1.evaluate(validation_dataset)

print(f"Categorical accuracy on the test set: {round(binary_acc1 * 100, 2)}%.")
print(f"Categorical accuracy on the validation set: {round(binary_acc2 * 100, 2)}%.")

# ***Save Model And Text Vectorizer***

In [ ]:
import pickle
# Save the model
model1.save("models/model.h5")

# Save the configuration of the text vectorizer
saved_text_vectorizer_config = text_vectorizer.get_config()
with open("models/text_vectorizer_config.pkl", "wb") as f:
    pickle.dump(saved_text_vectorizer_config, f)

# Save the weights of the text vectorizer (includes vocabulary and IDF weights)
saved_text_vectorizer_weights = text_vectorizer.get_weights()
with open("models/text_vectorizer_weights.pkl", "wb") as f:
    pickle.dump(saved_text_vectorizer_weights, f)

# Save the vocabulary (optional, as it's included in weights, but good for clarity)
with open("models/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# ***Load Model And Text Vectorizer***

In [ ]:
from tensorflow import keras
import pickle
from tensorflow.keras.layers import TextVectorization
import tensorflow as tf
import numpy as np

# Load the model
loaded_model = keras.models.load_model("models/model.h5")

# Load the configuration of the text vectorizer
with open("/content/models/text_vectorizer_config.pkl", "rb") as f:
    saved_text_vectorizer_config = pickle.load(f)

# Create a new TextVectorization layer with the saved configuration
loaded_text_vectorizer = TextVectorization.from_config(saved_text_vectorizer_config)

# Load the saved weights into the new TextVectorization layer
with open("/content/models/text_vectorizer_weights.pkl", "rb") as f:
    weights = pickle.load(f)
    loaded_text_vectorizer.set_weights(weights)

# Adapt the loaded vectorizer with the training data abstracts to initialize the lookup table
loaded_text_vectorizer.adapt(train_df['abstracts'].values)

# Load the vocabulary (optional)
with open("/content/models/vocab.pkl", "rb") as f:
    loaded_vocab = pickle.load(f)

In [31]:
import os

# Create the directory if it doesn't exist
if not os.path.exists("models"):
    os.makedirs("models")

In [32]:
# Load the vocabulary
# with open("models/vocab.pkl", "rb") as f:
#     loaded_vocab = pickle.load(f)

# ***Model Prediction***

In [33]:
def invert_multi_hot(encoded_labels):
    """Reverse a single multi-hot encoded label to a tuple of vocab terms."""
    hot_indices = np.argwhere(encoded_labels == 1.0)[..., 0]
    return np.take(loaded_vocab, hot_indices)

In [34]:
def predict_category(abstract, model, vectorizer, label_lookup):
    # Preprocess the abstract using the loaded text vectorizer
    preprocessed_abstract = vectorizer([abstract])

    # Make predictions using the loaded model
    predictions = model.predict(preprocessed_abstract)

    # Convert predictions to human-readable labels
    predicted_labels = label_lookup(np.round(predictions).astype(int)[0])

    return predicted_labels

In [35]:
# Save the entire model in the SavedModel format
model1.save("models/full_model.keras")

In [ ]:
# Load the entire model
loaded_full_model = keras.models.load_model("models/full_model.keras")

In [ ]:
loaded_full_model.summary()

Now you can use `loaded_full_model` directly for predictions, as it includes the text vectorization layer.

In [38]:
def predict_category(abstract, model, vectorizer, label_lookup):
    # Preprocess the abstract using the loaded text vectorizer
    preprocessed_abstract = vectorizer([abstract]).numpy().astype('float32')

    # Make predictions using the loaded model
    predictions = model.predict(preprocessed_abstract)

    # Convert predictions to human-readable labels
    predicted_labels = invert_multi_hot(np.round(predictions).astype(int)[0])

    return predicted_labels

In [ ]:
# Example usage with the loaded full model and loaded text vectorizer
new_abstract = '''Deep learning is a class of machine learning which performs much better on unstructured data. Deep learning techniques are outperforming current machine learning techniques. It enables computational models to learn features progressively from data at multiple levels. The popularity of deep learning amplified as the amount of data available increased as well as the advancement of hardware that provides powerful computers. This article comprises the evolution of deep learning, various approaches to deep learning, architectures of deep learning, methods, and applications.'''

# Preprocess the abstract using the loaded_text_vectorizer before passing it to the model
preprocessed_new_abstract = loaded_text_vectorizer([new_abstract]).numpy().astype('float32')

# Make predictions using the loaded_full_model with the preprocessed abstract
predictions = loaded_full_model.predict(preprocessed_new_abstract)

# Convert predictions to human-readable labels using the original vocab
predicted_labels = invert_multi_hot(np.round(predictions).astype(int)[0])

print("Predicted Categories:", predicted_labels)

# ***Recommendation System***

In [40]:
df.drop(columns = ["terms","abstracts"], inplace = True)

In [41]:
df.drop_duplicates(inplace= True)
df.reset_index(drop= True,inplace = True)

In [ ]:
pd.set_option('display.max_colwidth', None)
df

In [43]:
# !pip install -U -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = df['titles']
embeddings = model.encode(sentences)  #remove stop wrods, preprocessing and vectorisation

In [ ]:
embeddings

# ***Print the embeddings***

In [ ]:
c = 0
#This loop iterates over pairs of sentences and their corresponding embeddings.
#zip is used to iterate over both lists simultaneously.
for sentence, embedding in zip(sentences, embeddings):
    print("Sentence:", sentence)
    print("Embedding length:", len(embedding)) # list of floats
    print("")
    # Breaks out of the loop after printing information for the first 5 sentences.
    if c >=5:
        break
    c +=1

# ***Save File***

In [47]:
import pickle
import os

# Create the 'files' directory if it doesn't exist
if not os.path.exists('files'):
    os.makedirs('files')

# Saving sentences and corresponding embeddings
with open('files/embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

with open('files/sentences.pkl', 'wb') as f:
    pickle.dump(sentences, f)

with open('files/rec_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# ***Recommendation for similar papers***

In [48]:
# load save files
embeddings = pickle.load(open('files/embeddings.pkl','rb'))
sentences = pickle.load(open('files/sentences.pkl','rb'))
rec_model = pickle.load(open('files/rec_model.pkl','rb'))

In [49]:
import torch

def recommendation(input_paper):
    # Calculate cosine similarity scores between the embeddings of input_paper and all papers in the dataset.
    cosine_scores = util.cos_sim(embeddings, rec_model.encode(input_paper))

    # Get the indices of the top-k most similar papers based on cosine similarity.
    top_similar_papers = torch.topk(cosine_scores, dim=0, k=5, sorted=True)

    # Retrieve the titles of the top similar papers.
    papers_list = []
    for i in top_similar_papers.indices:
        papers_list.append(sentences[i.item()])

    return papers_list


In [ ]:
# exampel usage 1: (use this paper as input (Attention is All you Need))
input_paper = input("Enter the title of any paper you like")
recommend_papers = recommendation(input_paper)


print("We recommend to read this paper............")
print("=============================================")
for paper in recommend_papers:
    print(paper)

In [ ]:
# install this versions
import sentence_transformers
import tensorflow
import torch
print(torch.__version__)
print(sentence_transformers.__version__)
print(tensorflow.__version__)